# Retrieval of Financial Data from SEC Filings to Automate Metrics Collection

## Initial Setup

Import dependencies

In [26]:
from __future__ import annotations
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Literal, Set
import calendar
from datetime import datetime
import pandas as pd
import requests

Define the user agent details for the SEC API

In [27]:
USER_AGENT = "AirlineFinancialDashboard.com (michael.tricanowicz@live.com)"

## Extract standardized metrics into a table

### **Helper Functions**

Define functions to create period start and end date windows to filter filing retrieval

In [28]:
# Define function to create start and end dates
def define_period_dates(year: int, period: str) -> Tuple[datetime, datetime]:
    """
    period: 'FY' or 'Q1'/'Q2'/'Q3'/'Q4'
    Returns: (start_date, end_date) as datetimes covering that period.
    """
    if period == "FY":
        start_month = 1
        end_month = 12
    else:
        end_month = int(period[-1]) * 3
        start_month = end_month - 2
    start_day = 1
    end_day = calendar.monthrange(year, end_month)[1]
    # Create start and end date variables to constrain document scraping
    start_date = datetime(year, start_month, start_day)
    end_date = datetime(year, end_month, end_day)
    return start_date, end_date

# Define function to build date windows with labels when multiple years and/or periods are needed
def build_date_windows(years: Iterable[int], periods: Iterable[str]) -> List[Tuple[str, datetime, datetime]]:
    """
    Create a list of windows with labels, e.g.:
    [('2024-Q1', start, end), ('2024-FY', start, end), ...]
    """
    windows = []
    for y in years:
        for p in periods:
            s, e = define_period_dates(y, p)
            windows.append((f"{y}-{p}", s, e))
    # Sort for stability / readability
    windows.sort(key=lambda x: (x[1], x[0]))
    return windows

# Define function to assign window labels based on 'fp' and 'end' columns
def assign_window_label_by_fp_and_end(
    df: pd.DataFrame,
    windows: List[Tuple[str, datetime, datetime]],
    *,
    fp_col: str = "fp",
    start_col: str = "start",
    end_col: str = "end",
    out_col: str = "window_label",
) -> pd.DataFrame:
    """
    Assign a SINGLE window label to each row based on:
      - fp matches the window period
      - end is within the window [start, end]

    Special override for the common airline case:
      - Some filers tag both Q4 (3 months) and FY (12 months) facts as fp="FY".
      - In that case, use (start,end) to assign:
          Q4 if start=10/1/YYYY and end=12/31/YYYY
          FY if start=1/1/YYYY and end=12/31/YYYY

    For everything else:
      - Match by fp + end date inside window (instant facts match by end only).
    """
    if df.empty:
        return df

    x = df.copy()
    x[out_col] = None

    # normalize data types
    x[fp_col] = x[fp_col].astype(str).str.upper()
    x[start_col] = pd.to_datetime(x[start_col], errors="coerce")
    x[end_col] = pd.to_datetime(x[end_col], errors="coerce")

    # Build a quick lookup from year -> window label for Q4 and FY if present
    # e.g., {"2025": "2025-Q4"} etc.
    q4_label = {}
    fy_label = {}
    for label, _, _ in windows:
        y, p = label.split("-", 1)
        p = p.upper()
        if p == "Q4":
            q4_label[int(y)] = label
        elif p == "FY":
            fy_label[int(y)] = label

    # ---- 1) Hard override using exact start/end patterns (works even when fp is "FY")
    # Match only rows that have both start and end
    has_dates = x[start_col].notna() & x[end_col].notna()
    if has_dates.any():
        years = x.loc[has_dates, end_col].dt.year

        # Q4: Oct 1 to Dec 31
        is_q4 = (
            has_dates
            & (x[start_col].dt.month == 10) & (x[start_col].dt.day == 1)
            & (x[end_col].dt.month == 12) & (x[end_col].dt.day == 31)
            & (x[start_col].dt.year == x[end_col].dt.year)
        )

        # FY: Jan 1 to Dec 31
        is_fy = (
            has_dates
            & (x[start_col].dt.month == 1) & (x[start_col].dt.day == 1)
            & (x[end_col].dt.month == 12) & (x[end_col].dt.day == 31)
            & (x[start_col].dt.year == x[end_col].dt.year)
        )

        # assign labels where those windows exist in the selection
        for yr, lbl in q4_label.items():
            x.loc[is_q4 & (x[end_col].dt.year == yr), out_col] = lbl

        for yr, lbl in fy_label.items():
            # don't overwrite Q4 if already assigned
            m = is_fy & (x[end_col].dt.year == yr) & x[out_col].isna()
            x.loc[m, out_col] = lbl

    # ---- 2) Fallback: normal matching by fp + end inside window (or end-only for instant)
    # Only fill rows still unassigned
    unassigned = x[out_col].isna() & x[end_col].notna()
    if unassigned.any():
        has_start = x[start_col].notna()

        for label, w_start, w_end in windows:
            _, p = label.split("-", 1)
            p = p.upper()

            # duration facts overlap
            dur = (
                unassigned
                & has_start
                & (x[fp_col] == p)
                & (x[start_col] <= w_end)
                & (x[end_col] >= w_start)
            )

            # instant facts end inside window
            inst = (
                unassigned
                & (~has_start)
                & (x[fp_col] == p)
                & (x[end_col] >= w_start)
                & (x[end_col] <= w_end)
            )

            x.loc[dur | inst, out_col] = label
            unassigned = x[out_col].isna() & x[end_col].notna()
            if not unassigned.any():
                break

    return x

# Define function to keep only rows where window_label matches 'fy-fp'
def keep_only_rows_where_window_matches_fy_fp(
    df: pd.DataFrame,
    *,
    fy_col: str = "fy",
    fp_col: str = "fp",
    window_col: str = "window_label",
) -> pd.DataFrame:
    """
    Keep only rows where window_label == f'{fy}-{fp}'.

    This removes:
    - FY rows being retained when you wanted Q4
    - prior-year comparatives that slipped through odd fy/fp reporting
    """
    if df.empty or window_col not in df.columns:
        return df

    x = df.copy()
    x[fp_col] = x[fp_col].astype(str).str.upper()
    x[fy_col] = pd.to_numeric(x[fy_col], errors="coerce").astype("Int64")

    x["fy_fp_label"] = x.apply(
        lambda r: f"{int(r[fy_col])}-{r[fp_col]}" if pd.notna(r[fy_col]) and pd.notna(r[fp_col]) else None,
        axis=1
    )

    x = x[x[window_col].notna() & (x[window_col] == x["fy_fp_label"])]
    return x.drop(columns=["fy_fp_label"])


Define functions to create maps of airlines of interest and their CIKs for filing retrieval

In [ ]:
# Define function to fetch ticker to CIK mapping
def fetch_ticker_to_cik_map(user_agent: str) -> Dict[str, str]:
    """
    Download SEC's company_tickers.json ONCE and build {ticker -> 10-digit CIK}.

    Why:
    - Avoids repeated downloads per ticker
    - Faster, more polite, fewer rate-limit issues
    """
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": user_agent}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    ticker_to_cik = {}
    for item in data.values():
        ticker = item["ticker"].upper()
        cik10 = str(item["cik_str"]).zfill(10)
        ticker_to_cik[ticker] = cik10
    return ticker_to_cik

# Define function to build airline CIK dictionary
def build_airline_cik_dict(airlines: List[str], ticker_to_cik: Dict[str, str]) -> Dict[str, str]:
    """
    Convert a list of tickers into {ticker: cik10}. Skips unknown tickers.
    """
    out = {}
    for t in airlines:
        t_up = t.upper()
        cik = ticker_to_cik.get(t_up)
        if cik:
            out[t_up] = cik
    return out

# Fetch and define the ticker mapping dictionary
ticker_map = fetch_ticker_to_cik_map(USER_AGENT)

Define function to normalize the CIK format to ensure it is compatible with the SEC API

In [30]:
def normalize_cik(cik: str | int) -> str:
    """
    Convert CIK to a zero-padded 10-digit string.

    Why:
    - SEC endpoints typically want 'CIK##########' where the CIK is 10 digits.
    - Users commonly provide CIKs with fewer digits or with formatting.
    """
    s = re.sub(r"[^\d]", "", str(cik))  # keep digits only
    return s.zfill(10)

### **SEC client Setup**

In [31]:
# -----------------------------
# SEC client with:
# - descriptive User-Agent
# - polite throttling (avoid hammering SEC)
# - retry with backoff on transient errors
# -----------------------------
@dataclass
class SecClient:
    """
    Polite SEC HTTP client:
    - descriptive User-Agent required by SEC
    - throttling
    - retry/backoff for transient 429/5xx
    """
    user_agent: str
    min_interval_sec: float = 0.15  # ~6-7 requests/sec. Safely under typical guidance of less than 10 requests per second.
    timeout: int = 30
    max_retries: int = 5

    # SEC endpoints used
    base_submissions: str = "https://data.sec.gov/submissions"
    base_xbrl: str = "https://data.sec.gov/api/xbrl"
    base_archives: str = "https://www.sec.gov/Archives"

    def __post_init__(self) -> None:
        # Use a requests.Session for connection pooling
        self.sess = requests.Session()

        # User-Agent is REQUIRED by SEC fair-access policy; include contact.
        self.sess.headers.update({
            "User-Agent": self.user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Accept": "application/json,text/html,text/plain,*/*",
        })

        # Track last request time to enforce min interval between requests
        self._last_ts = 0.0

    def _throttle(self) -> None:
        """
        Enforce a minimum time gap between requests.

        Why:
        - Keeps you compliant with SEC fair access expectations.
        - Reduces 429 "Too Many Requests" errors.
        """
        dt = time.time() - self._last_ts
        if dt < self.min_interval_sec:
            time.sleep(self.min_interval_sec - dt)

    def get(self, url: str, *, stream: bool = False) -> requests.Response:
        """
        GET with throttling + retry + exponential backoff.

        Retries for:
        - 429 rate limit
        - 5xx server errors
        """
        for attempt in range(1, self.max_retries + 1):
            self._throttle()
            try:
                r = self.sess.get(url, timeout=self.timeout, stream=stream)
                self._last_ts = time.time()

                # Retry on common transient failures
                if r.status_code in (429, 500, 502, 503, 504):
                    time.sleep(min(2 ** attempt, 30))
                    continue

                r.raise_for_status()
                return r

            except requests.RequestException:
                if attempt == self.max_retries:
                    raise
                time.sleep(min(2 ** attempt, 30))

        raise RuntimeError("Unreachable: exceeded retry loop")

### **Retrieve financial metrics using XBRL companyfacts**

The XBRL companyfacts endpoint returns all facts reported by a company, including:
- US-GAAP concepts (standardized)
- DEI / other taxonomies
- Company extension concepts (often where ASM/RPM live)

We’ll implement:
- Fetch companyfacts JSON
- Extract time series for specific concepts
- Choose the latest fact per fiscal period (FY/Q)
- Output a tidy dataframe for multi-airline visualization

In [32]:
def companyfacts(client: SecClient, cik: str | int) -> Dict[str, Any]:
    """
    Fetch all XBRL facts for the company.

    Why:
    - This is the best "structured metrics" source.
    - It includes both standard US-GAAP tags and company extensions.
    """
    cik10 = normalize_cik(cik)
    url = f"{client.base_xbrl}/companyfacts/CIK{cik10}.json"
    return client.get(url).json()

Utility Functions

In [33]:
# Iterate and extract all concepts in companyfacts
def iter_all_concepts(cf: Dict[str, Any]) -> List[Tuple[str, str, Dict[str, Any]]]:
    """
    Iterate across every taxonomy and concept in companyfacts.

    Why:
    - Airline KPIs like ASM/RPM may not be in us-gaap.
    - They often appear in an extension taxonomy (company-specific prefix).
    """
    out = []
    facts = cf.get("facts", {})
    for tax, concepts in facts.items():
        for concept, payload in concepts.items():
            out.append((tax, concept, payload))
    return out

# Enable keyword-based concept discovery
def find_concepts_by_keywords(cf: Dict[str, Any], keywords: List[str]) -> pd.DataFrame:
    """
    Discover candidate concepts by keyword matching on:
    - taxonomy:concept name
    - label (human-readable)

    Why:
    - ASM/RPM/Profit Sharing may be in extension concepts with non-obvious names.
    - This discovery step helps you build a mapping per airline.
    """
    rows = []
    kw = [k.lower() for k in keywords]

    for tax, concept, payload in iter_all_concepts(cf):
        label = (payload.get("label") or "").lower()
        c_lower = concept.lower()

        # Combine taxonomy, concept, label into searchable text
        hay = f"{tax}:{c_lower} {label}"

        if any(k in hay for k in kw):
            # units show you whether this is USD, shares, pure, etc.
            units = list((payload.get("units") or {}).keys())
            rows.append({
                "taxonomy": tax,
                "concept": concept,
                "label": payload.get("label"),
                "units": units,
            })

    return pd.DataFrame(rows).sort_values(["taxonomy", "concept"])

cf = companyfacts(client, airline_ciks["AAL"])


Extract + filter facts (the critical “no-duplicates” core)

In [34]:
# Extract the time series for a given (taxonomy, concept)
def extract_concept_timeseries(
    cf: Dict[str, Any],
    taxonomy: str,
    concept: str,
    unit: Optional[str] = None,
) -> pd.DataFrame:
    """
    Extract tidy facts for one (taxonomy, concept) from companyfacts.

    Key columns returned:
    - val (value)
    - start, end (period boundaries)  <-- essential to keep FY vs Q4 distinct when same end date
    - fy, fp (fiscal year, fiscal period)
    - form, filed, accn, frame
    """
    try:
        payload = cf["facts"][taxonomy][concept]
    except KeyError:
        return pd.DataFrame()

    units = payload.get("units", {}) or {}
    if not units:
        return pd.DataFrame()

    # choose a unit
    if unit is None:
        unit = "USD" if "USD" in units else next(iter(units.keys()), None)
    if not unit or unit not in units:
        return pd.DataFrame()

    rows = []
    for ob in units[unit]:
        rows.append({
            "taxonomy": taxonomy,
            "concept": concept,
            "label": payload.get("label"),
            "unit": unit,
            "val": ob.get("val"),
            "start": ob.get("start"),
            "end": ob.get("end"),
            "fy": ob.get("fy"),
            "fp": ob.get("fp"),
            "form": ob.get("form"),
            "filed": ob.get("filed"),
            "accn": ob.get("accn"),
            "frame": ob.get("frame"),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["start"] = pd.to_datetime(df["start"], errors="coerce")
    df["end"] = pd.to_datetime(df["end"], errors="coerce")
    df["filed"] = pd.to_datetime(df["filed"], errors="coerce")
    df["fp"] = df["fp"].astype(str).str.upper()
    df["fy"] = pd.to_numeric(df["fy"], errors="coerce").astype("Int64")

    # Keep only records that have an end date (essential for period selection)
    df = df.dropna(subset=["end"])
    return df

# Filter the results on the selected periods
def filter_to_selected_periods_by_fp_and_window(
    df: pd.DataFrame,
    windows: List[Tuple[str, datetime, datetime]],
) -> pd.DataFrame:
    """
    This is the simple, robust selector that avoids the “comparatives in same filing” problem:

    Keep a row if:
      - its fp matches the window period (Q1/Q2/Q3/Q4/FY)
      - its start and end date falls inside that window

    No matched_windows. No overlap confusion.
    """
    if df.empty or not windows:
        return df

    x = df.copy()
    keep = pd.Series(False, index=x.index)

    for label, start, end in windows:
        y, p = label.split("-", 1)
        y = int(y)
        p = p.upper()
        if x["start"].notna().any():
            keep = keep | ((x["fy"] == y) & (x["fp"] == p) & (x["start"] >= start) & (x["start"] < end) & (x["end"] > start) & (x["end"] <= end))
        else:
            keep = keep | ((x["fy"] == y) & (x["fp"] == p) & (x["end"] > start) & (x["end"] <= end))

    return x[keep]

# Deduplicate repeated values
def dedupe_keep_latest_per_unique_period(df: pd.DataFrame) -> pd.DataFrame:
    """
    Deduplicate while preserving FY vs Q4.

    Key:
    - Q4 (3 months) and FY (12 months) can share the same end date in a 10-K.
    - Including start in the grouping keeps them distinct.
    """
    if df.empty:
        return df

    x = df.copy()

    # Prefer “real” annual/quarterly forms if duplicates exist
    x["form_pref"] = x["form"].isin(["10-Q", "10-K"]).astype(int)

    group_cols = ["taxonomy", "concept", "unit", "fp", "end"]
    if x["start"].notna().any():
        group_cols.append("start")  # preserves FY vs Q4 when both exist

    # Sort so the last row per group is the “best” one to keep
    x = x.sort_values(group_cols + ["form_pref", "filed"])
    x = x.groupby(group_cols, as_index=False).tail(1)
    return x.drop(columns=["form_pref"])

Optional: compute Q4 if missing (flow metrics only)

In [35]:
def _pick_best_row(g: pd.DataFrame) -> pd.Series:
    """
    Pick the best single row from a set of duplicates for the same period.
    Preference:
      1) non-null value
      2) form: 10-Q/10-K > others > COMPUTED
      3) latest filed
    """
    x = g.copy()
    x["filed"] = pd.to_datetime(x["filed"], errors="coerce")
    x["form_rank"] = x["form"].map({"10-Q": 3, "10-K": 3, "COMPUTED": 0}).fillna(1)
    x["has_val"] = x["value"].notna().astype(int)
    x = x.sort_values(["has_val", "form_rank", "filed"])
    return x.iloc[-1]

def compute_q4_if_missing(
    df: pd.DataFrame,
    *,
    flow_metrics: set[str] | None = None,
    require_same_period_end: bool = True,
) -> pd.DataFrame:
    """
    Compute Q4 for flow metrics when missing using:
        Q4 = FY - (Q1 + Q2 + Q3)

    Assumes df has at least:
      ticker, metric, fy, fp, value, period_end, form, filed

    Key protections:
    - Uses only QTD quarters (Q1/Q2/Q3) and FY rows (not YTD duplicates) by:
        selecting ONE row per (ticker, metric, fy, fp) using _pick_best_row
      You should ideally have already filtered out YTD vs QTD earlier, but this
      still works even if some duplicates remain.
    - If require_same_period_end=True, it uses the FY period_end as the Q4 end date.

    Returns df with computed Q4 rows appended (form='COMPUTED').
    """
    if df.empty:
        return df

    if flow_metrics is None:
        flow_metrics = {"TotalRevenue", "TotalExpenses", "NetIncome", "PassengerRevenue", "ProfitSharing"}

    x = df.copy()
    x["fy"] = pd.to_numeric(x["fy"], errors="coerce").astype("Int64")
    x["fp"] = x["fp"].astype(str).str.upper()
    x["period_end"] = pd.to_datetime(x["period_end"], errors="coerce")
    x["value"] = pd.to_numeric(x["value"], errors="coerce")

    # We'll compute per (ticker, metric, fy)
    new_rows = []

    for (ticker, metric, fy), g in x.groupby(["ticker", "metric", "fy"], dropna=True):
        if metric not in flow_metrics:
            continue

        # Already has Q4 → skip
        if (g["fp"] == "Q4").any():
            continue

        # Need FY, Q1, Q2, Q3
        needed = {"FY", "Q1", "Q2", "Q3"}
        have = set(g["fp"].dropna().unique())
        if not needed.issubset(have):
            continue

        # Pick a single "best" row for each required fp
        chosen = {}
        for p in ["FY", "Q1", "Q2", "Q3"]:
            chosen[p] = _pick_best_row(g[g["fp"] == p])

        # Ensure numeric values exist
        if any(pd.isna(chosen[p]["value"]) for p in ["FY", "Q1", "Q2", "Q3"]):
            continue

        q4_val = chosen["FY"]["value"] - (chosen["Q1"]["value"] + chosen["Q2"]["value"] + chosen["Q3"]["value"])

        # Use FY period_end as Q4 end date (standard)
        q4_end = chosen["FY"]["period_end"] if require_same_period_end else g["period_end"].max()

        # Carry identity fields from FY row (taxonomy/concept/unit should match)
        fy_row = chosen["FY"]

        new_rows.append({
            "ticker": ticker,
            "cik": fy_row.get("cik"),
            "airline": fy_row.get("airline"),
            "metric": metric,
            "taxonomy": fy_row.get("taxonomy"),
            "concept": fy_row.get("concept"),
            "unit": fy_row.get("unit"),
            "value": q4_val,
            "fy": int(fy),
            "fp": "Q4",
            "period_end": q4_end,
            "form": "COMPUTED",
            "filed": pd.NaT,
            "accession": None,
            # Optional columns if present in your df
            "start": pd.NaT,
            "frame": None,
        })

    if not new_rows:
        return x

    out = pd.concat([x, pd.DataFrame(new_rows)], ignore_index=True)

    # Final cleanup: ensure consistent dtypes
    out["fy"] = pd.to_numeric(out["fy"], errors="coerce").astype("Int64")
    out["fp"] = out["fp"].astype(str).str.upper()
    out["period_end"] = pd.to_datetime(out["period_end"], errors="coerce")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")

    return out

### **Define your metric mapping strategy**

GAAP metrics (usually standardized)
- A good starting point:
- Total Revenue → us-gaap:Revenues
- Net Income → us-gaap:NetIncomeLoss
- Total Expenses → (varies; sometimes OperatingExpenses, sometimes CostsAndExpenses)
- Long Term Debt → LongTermDebtNoncurrent (often present, but companies can vary)

Airline KPIs (often extensions)
- RPM / ASM / Passenger Revenue / Profit Sharing may be extension concepts
- We use find_concepts_by_keywords() to discover them per airline, then store a mapping.

In [52]:
# Define metric mapping to the companyfacts taxonomy and concepts: metric_name -> (taxonomy, concept, unit)
GAAP_MAP: Dict[str, Tuple[str, str, Optional[str]]] = {
    "TotalRevenue": ("us-gaap", "Revenues", "USD"),
    "PassengerRevenue": ("us-gaap", "PassengerRevenue", "USD"),
    "NetIncome": ("us-gaap", "NetIncomeLoss", "USD"),
    "TotalExpenses": ("us-gaap", "CostsAndExpenses", "USD"),
}

# Debt often vary; we’ll handle as fallback candidates
DEBT_CANDIDATES: List[Tuple[str, str, Optional[str]]] = [
    ("us-gaap", "LongTermDebt", "USD"),
    ("us-gaap", "LongTermDebtCurrent", "USD"),
    ("us-gaap", "LongTermDebtNoncurrent", "USD")
]

# Airline KPIs: YOU WILL LIKELY NEED TO FILL THESE IN AFTER DISCOVERY.
# Set None or omit metrics you haven’t mapped yet.
# Example structure shown; concept/taxonomy vary by filer.
AIRLINE_KPI_MAP_BY_TICKER: Dict[str, Dict[str, Tuple[str, str, Optional[str]]]] = {
    # "AAL": {
    #     "PassengerRevenue": ("aal", "PassengerRevenue", "USD"),
    #     "RPM": ("aal", "RevenuePassengerMiles", None),
    #     "ASM": ("aal", "AvailableSeatMiles", None),
    #     "ProfitSharing": ("aal", "ProfitSharing", "USD"),
    # },
}

In [37]:

def extract_first_available(
    cf: Dict[str, Any],
    candidates: List[Tuple[str, str, Optional[str]]],
) -> pd.DataFrame:
    """
    Try a list of (taxonomy, concept, unit) and return the first non-empty extracted DF.
    """
    for tax, concept, unit in candidates:
        ts = extract_concept_timeseries(cf, tax, concept, unit)
        if not ts.empty:
            return ts
    return pd.DataFrame()


In [38]:
# Define function to build airline metrics table by fiscal year and period
def build_metrics_table(
    client: SecClient,
    airlines: List[Dict[str, str]],
    windows: List[Tuple[str, datetime, datetime]],
    *,
    include_gaap: bool = True,
    include_debt: bool = True,
    include_airline_kpis: bool = True,
    compute_missing_q4: bool = False,
) -> pd.DataFrame:
    """
    End-to-end builder:
    - fetch companyfacts per airline
    - extract metrics
    - filter to selected (periods) by fp + end date
    - dedupe correctly
    - return tidy table suitable for visualization
    """
    rows: List[Dict[str, Any]] = []

    for a in airlines:
        ticker = a["ticker"]
        cik = a["cik"]
        name = a.get("name", ticker)

        cf = companyfacts(client, cik)
##################################################################################################################################################################################
        # ---- Primary metrics
        if include_gaap:
            for metric, (tax, concept, unit) in GAAP_MAP.items():
                ts = extract_concept_timeseries(cf, tax, concept, unit)
                ts = filter_to_selected_periods_by_fp_and_window(ts, windows)
                ts = assign_window_label_by_fp_and_end(ts, windows, fp_col="fp", end_col="end")
                ts = dedupe_keep_latest_per_unique_period(ts)

                if ts.empty:
                    continue

                for _, r in ts.iterrows():
                    rows.append({
                        "ticker": ticker,
                        "cik": normalize_cik(cik),
                        "airline": name,
                        "metric": metric,
                        "taxonomy": r["taxonomy"],
                        "concept": r["concept"],
                        "unit": r["unit"],
                        "value": r["val"],
                        "fy": r["fy"],
                        "fp": r["fp"],
                        "start": r["start"],
                        "period_end": r["end"],
                        "form": r["form"],
                        "filed": r["filed"],
                        "accession": r["accn"],
                        "window_label": r.get("window_label"),
                    })
##################################################################################################################################################################################        
        # ---- Debt (period-by-period fallback)
        # Behavior per (fp, end, unit):
        # 1) Prefer us-gaap:LongTermDebt if present for that period
        # 2) Else compute = LongTermDebtCurrent + LongTermDebtNoncurrent (sum what exists)
        # This handles concept changes across years (e.g., UAL stops reporting LongTermDebt in 2015).

        if include_debt:
            # 1) Extract each concept independently
            ts_total = extract_concept_timeseries(cf, "us-gaap", "LongTermDebt", "USD")
            ts_curr  = extract_concept_timeseries(cf, "us-gaap", "LongTermDebtCurrent", "USD")
            ts_non   = extract_concept_timeseries(cf, "us-gaap", "LongTermDebtNoncurrent", "USD")

            # 2) Apply identical filter/label/dedupe steps to each series
            def _prep(ts: pd.DataFrame) -> pd.DataFrame:
                if ts.empty:
                    return ts
                ts = filter_to_selected_periods_by_fp_and_window(ts, windows)
                ts = assign_window_label_by_fp_and_end(ts, windows, fp_col="fp", end_col="end")
                ts = dedupe_keep_latest_per_unique_period(ts)
                return ts

            ts_total = _prep(ts_total)
            ts_curr  = _prep(ts_curr)
            ts_non   = _prep(ts_non)

            # 3) Index them by period key so we can choose per-period
            # Use a stable key for balance sheet facts: (fp, end, unit, window_label)
            # window_label helps when you select multiple years/periods at once.
            def _key_df(ts: pd.DataFrame, value_col: str) -> pd.DataFrame:
                if ts.empty:
                    return ts
                x = ts.copy()
                x["fp"] = x["fp"].astype(str).str.upper()
                x["end"] = pd.to_datetime(x["end"], errors="coerce")
                x["unit"] = x["unit"].astype(str)
                # keep window_label if present (may be None)
                x["window_label"] = x.get("window_label")
                return x

            ts_total = _key_df(ts_total, "val")
            ts_curr  = _key_df(ts_curr,  "val")
            ts_non   = _key_df(ts_non,   "val")

            # Build dicts keyed by (fp, end, unit, window_label)
            def _to_dict(ts: pd.DataFrame) -> dict:
                d = {}
                if ts.empty:
                    return d
                for _, r in ts.iterrows():
                    k = (r.get("fp"), r.get("end"), r.get("unit"), r.get("window_label"))
                    d[k] = r
                return d

            total_map = _to_dict(ts_total)
            curr_map  = _to_dict(ts_curr)
            non_map   = _to_dict(ts_non)

            # Union of all keys we have any debt info for
            all_keys = set(total_map) | set(curr_map) | set(non_map)

            # 4) Emit one LongTermDebt row per key
            for k in sorted(all_keys, key=lambda t: (str(t[0]), pd.Timestamp(t[1]) if pd.notna(t[1]) else pd.Timestamp.min)):
                fp, end_dt, unit, win = k

                if k in total_map:
                    r = total_map[k]
                    value = r.get("val")
                    concept_used = "LongTermDebt"
                    provenance_form = r.get("form")
                    provenance_filed = r.get("filed")
                    provenance_accn = r.get("accn")
                    fy = r.get("fy")
                    start = r.get("start")
                    taxonomy = r.get("taxonomy", "us-gaap")
                else:
                    r_c = curr_map.get(k)
                    r_n = non_map.get(k)

                    v_c = r_c.get("val") if r_c is not None else None
                    v_n = r_n.get("val") if r_n is not None else None

                    # If neither exists, skip (shouldn't happen due to key union, but safe)
                    if v_c is None and v_n is None:
                        continue

                    # Sum what exists
                    value = (0 if v_c is None else v_c) + (0 if v_n is None else v_n)

                    if v_c is not None and v_n is not None:
                        concept_used = "LongTermDebtCurrent+LongTermDebtNoncurrent"
                    elif v_c is not None:
                        concept_used = "LongTermDebtCurrent"
                    else:
                        concept_used = "LongTermDebtNoncurrent"

                    # Choose provenance: whichever side has later filed date (if both)
                    def _dt(x):
                        return pd.to_datetime(x, errors="coerce") if x is not None else pd.NaT

                    filed_c = _dt(r_c.get("filed")) if r_c is not None else pd.NaT
                    filed_n = _dt(r_n.get("filed")) if r_n is not None else pd.NaT

                    pick = r_c if (pd.isna(filed_n) or (not pd.isna(filed_c) and filed_c >= filed_n)) else r_n

                    provenance_form = pick.get("form") if pick is not None else None
                    provenance_filed = pick.get("filed") if pick is not None else pd.NaT
                    provenance_accn = pick.get("accn") if pick is not None else None

                    # Coalesce common fields
                    fy = (r_c.get("fy") if r_c is not None else None) or (r_n.get("fy") if r_n is not None else None)
                    start = (r_c.get("start") if r_c is not None else None) or (r_n.get("start") if r_n is not None else None)
                    taxonomy = (r_c.get("taxonomy") if r_c is not None else None) or (r_n.get("taxonomy") if r_n is not None else None) or "us-gaap"

                rows.append({
                    "ticker": ticker,
                    "cik": normalize_cik(cik),
                    "airline": name,
                    "metric": "LongTermDebt",
                    "taxonomy": taxonomy,
                    "concept": concept_used,
                    "unit": unit,
                    "value": value,
                    "fy": fy,
                    "fp": fp,
                    "start": start,          # likely NaT for balance sheet facts
                    "period_end": end_dt,
                    "form": provenance_form,
                    "filed": provenance_filed,
                    "accession": provenance_accn,
                    "window_label": win,
                })
##################################################################################################################################################################################        
        # ---- Airline KPIs (optional, needs mapping)
        if include_airline_kpis and ticker in AIRLINE_KPI_MAP_BY_TICKER:
            for metric, (tax, concept, unit) in AIRLINE_KPI_MAP_BY_TICKER[ticker].items():
                ts = extract_concept_timeseries(cf, tax, concept, unit)
                ts = filter_to_selected_periods_by_fp_and_window(ts, windows)
                ts = assign_window_label_by_fp_and_end(ts, windows, fp_col="fp", end_col="end")
                ts = dedupe_keep_latest_per_unique_period(ts)

                if ts.empty:
                    continue

                for _, r in ts.iterrows():
                    rows.append({
                        "ticker": ticker,
                        "cik": normalize_cik(cik),
                        "airline": name,
                        "metric": metric,
                        "taxonomy": r["taxonomy"],
                        "concept": r["concept"],
                        "unit": r["unit"],
                        "value": r["val"],
                        "fy": r["fy"],
                        "fp": r["fp"],
                        "start": r["start"],
                        "period_end": r["end"],
                        "form": r["form"],
                        "filed": r["filed"],
                        "accession": r["accn"],
                        "window_label": r.get("window_label"),
                    })
##################################################################################################################################################################################
    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["period_end"] = pd.to_datetime(df["period_end"], errors="coerce")
    df["filed"] = pd.to_datetime(df["filed"], errors="coerce")
    df["fp"] = df["fp"].astype(str).str.upper()

    # Optional: compute Q4 for flow metrics if missing
    if compute_missing_q4:
        df = compute_q4_if_missing(df)

    # Final sort + final dedupe safety net:
    # This dedupe should remove nothing if upstream logic is correct, but it guards against edge cases.
    df = df.sort_values(["ticker", "metric", "fy", "fp", "period_end", "filed"])
    df = df.drop_duplicates(
        subset=["ticker", "metric", "fp", "period_end", "start", "value", "taxonomy", "concept", "unit"],
        keep="last"
    ).reset_index(drop=True)

    return df

### **Build Metrics Table**

In [53]:
# Set up the SEC client using the previously defined user agent
client = SecClient(user_agent=USER_AGENT)

# Define the airlines(s) of interest and generate their CIK mapping
airlines = ["AAL", "UAL", "DAL", "LUV"]
airline_ciks = build_airline_cik_dict(airlines, ticker_map)
airlines = [{"ticker": t, "cik": cik, "name": t} for t, cik in airline_ciks.items()]

# Define the period(s) of interest and generate filter windows
years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
periods = ["FY", "Q1", "Q2", "Q3", "Q4"]
windows = build_date_windows(years, periods)

# Generate metrics table
metrics_df = build_metrics_table(
    client,
    airlines=airlines,
    windows=windows,
    include_gaap=True,
    include_debt=True,           # will attempt to retrieve LongTermDebt metric if available
    include_airline_kpis=False,   # will only add KPIs for tickers in AIRLINE_KPI_MAP_BY_TICKER
    compute_missing_q4=False,    # compute Q4 for flow metrics if not provided
)

# Display metrics table
# Set display options to show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(metrics_df)
# Reset display options
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

,ticker,cik,airline,metric,taxonomy,concept,unit,value,fy,fp,start,period_end,form,filed,accession,window_label
0,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,20566000000,2018,FY,NaT,2018-12-31,10-K,2019-02-25,0000006201-19-000009,2018-FY
1,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,20896000000,2019,FY,NaT,2019-12-31,10-K,2020-02-19,0000006201-20-000023,2019-FY
2,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,20065000000,2019,Q1,NaT,2019-03-31,10-Q,2019-04-26,0000006201-19-000023,2019-Q1
3,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,21222000000,2019,Q2,NaT,2019-06-30,10-Q,2019-07-25,0000006201-19-000053,2019-Q2
4,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,21055000000,2019,Q3,NaT,2019-09-30,10-Q,2019-10-24,0000006201-19-000067,2019-Q3
5,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,29324000000,2020,FY,NaT,2020-12-31,10-K,2021-02-17,0000006201-21-000014,2020-FY
6,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,21033000000,2020,Q1,NaT,2020-03-31,10-Q,2020-04-30,0000006201-20-000066,2020-Q1
7,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,28193000000,2020,Q2,NaT,2020-06-30,10-Q,2020-07-23,0000006201-20-000089,2020-Q2
8,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,29593000000,2020,Q3,NaT,2020-09-30,10-Q,2020-10-22,0000006201-20-000101,2020-Q3
9,AAL,0000006201,AAL,LongTermDebt,us-gaap,LongTermDebt,USD,35008000000,2021,FY,NaT,2021-12-31,10-K,2022-02-22,0000006201-22-000026,2021-FY


In [46]:
print(pd.get_option('display.max_rows'))
print(pd.get_option('display.max_columns'))

60
20


In [54]:
concepts = find_concepts_by_keywords(companyfacts(client, "0000100517"), ["revenue", "passenger", "sales"])
concepts#[concepts["taxonomy"] == "us-gaap"]

,taxonomy,concept,label,units
0,us-gaap,BusinessAcquisitionsProFormaRevenue,"Business Acquisition, Pro Forma Revenue",[USD]
1,us-gaap,CargoAndFreightRevenue,Cargo and Freight Revenue (Deprecated 2018-01-31),[USD]
2,us-gaap,DeferredRevenue,Deferred Revenue,[USD]
3,us-gaap,DefinedBenefitPlanPurchasesSalesAndSettlements,"Defined Benefit Plan, Plan Assets Level 3 Reco...",[USD]
4,us-gaap,IncreaseDecreaseInDeferredAirTrafficRevenue,Increase (Decrease) in Deferred Air Traffic Re...,[USD]
5,us-gaap,NewAccountingPronouncementOrChangeInAccounting...,New Accounting Pronouncement or Change in Acco...,[USD]
6,us-gaap,OtherComprehensiveIncomeAvailableforsaleSecuri...,"Other Comprehensive Income (Loss), Available-f...",[USD]
7,us-gaap,OtherSalesRevenueNet,"Other Revenue, Net (Deprecated 2018-01-31)",[USD]
8,us-gaap,PassengerRevenue,Passenger Revenue (Deprecated 2018-01-31),[USD]
9,us-gaap,PassengerRevenueMainline,"Passenger Revenue, Mainline (Deprecated 2018-0...",[USD]


In [57]:
for tax in companyfacts(client, "0000100517")["facts"]:
    print(tax)

dei
invest
us-gaap
srt


In [67]:
tax_con = {}
for tax in companyfacts(client, "0000006201")["facts"]:
    tax_con[tax] = []
    for concept in companyfacts(client, "0000006201")["facts"][tax]:
        tax_con[tax].append(concept)
tax_con

{'dei': ['EntityCommonStockSharesOutstanding',
  'EntityNumberOfEmployees',
  'EntityPublicFloat'],
 'invest': ['DerivativeNonmonetaryNotionalAmount'],
 'us-gaap': ['AccountsPayableCurrent',
  'AccountsPayableRelatedPartiesCurrent',
  'AccruedLiabilitiesCurrent',
  'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
  'AccumulatedOtherComprehensiveIncomeLossNetOfTax',
  'AdditionalPaidInCapitalCommonStock',
  'AdjustmentForAmortization',
  'AdjustmentsRelatedToTaxWithholdingForShareBasedCompensation',
  'AdjustmentsToAdditionalPaidInCapitalEquityComponentOfConvertibleDebt',
  'AdjustmentsToAdditionalPaidInCapitalIncomeTaxDeficiencyFromShareBasedCompensation',
  'AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue',
  'AdjustmentsToAdditionalPaidInCapitalWarrantIssued',
  'AdvertisingExpense',
  'AircraftMaintenanceMaterialsAndRepairs',
  'AircraftRental',
  'AirlineCapacityPurchaseArrangements',
  'AirlineRelatedInven

In [66]:
companyfacts(client, "0000006201")["facts"]["us-gaap"]["NonoperatingIncomeExpense"]

{'label': 'Nonoperating Income (Expense)',
 'description': 'The aggregate amount of income or expense from ancillary business-related activities (that is to say, excluding major activities considered part of the normal operations of the business).',
 'units': {'USD': [{'start': '2008-01-01',
    'end': '2008-12-31',
    'val': -229000000,
    'accn': '0000950123-11-014726',
    'fy': 2010,
    'fp': 'FY',
    'form': '10-K',
    'filed': '2011-02-16',
    'frame': 'CY2008'},
   {'start': '2009-01-01',
    'end': '2009-06-30',
    'val': -345000000,
    'accn': '0000950123-10-066894',
    'fy': 2010,
    'fp': 'Q2',
    'form': '10-Q',
    'filed': '2010-07-21'},
   {'start': '2009-04-01',
    'end': '2009-06-30',
    'val': -164000000,
    'accn': '0000950123-10-066894',
    'fy': 2010,
    'fp': 'Q2',
    'form': '10-Q',
    'filed': '2010-07-21',
    'frame': 'CY2009Q2'},
   {'start': '2009-01-01',
    'end': '2009-09-30',
    'val': -540000000,
    'accn': '0000950123-10-094605',
  